# LLM_network




**notes**: 
* variables that you need to update are highlighted in <font color="red">*red*</font>
* variables that you can update but are not essential are highlighted in * <font color="orange">*orange*</font>

In [ ]:
# Import libraries
import sys
import os
import time
import pandas as pd
import docx2txt 

### Define directories


Variables to update:
* <font color="red">*root_dir*</font>
* <font color="red">*report2_dir*</font>
* <font color="red">*target_data_dir*</font>


In [4]:
# Define the project root directory
root_dir = '/Users/giorgiobolchi2/Documents/GitHub/jrc-egd/LLM/'

# File paths
report2_dir = '/Users/giorgiobolchi2/Documents/GitHub/jrc-egd/LLM/Data/REPORT_2/access_20250417'
target_data_dir  = '/Users/giorgiobolchi2/Documents/GitHub/jrc-egd/LLM/Data/targets_data_150.csv'

# Define absolute python path
sys.path.insert(0, root_dir) 

### Import data & functions


Variables you can change:
* <font color="orange">*filenames of each chapters*</font>

In [5]:

# Import target data
target_data_150 = pd.read_csv('target_data_dir', sep=";")  #target list susbet as in report 1


# Import chapters of report2   
report2 = { 
    'chapter1': docx2txt.process(f'{report2_dir}/Chapter1 - Setting the scene.docx'), # extract text from docx files
    'chapter2': docx2txt.process(f'{report2_dir}/Chapter2 - Challenges and enablers.docx'),
    'chapter3': docx2txt.process(f'{report2_dir}/Chapter3 - Environmental impacts of future scenarios - RP 2025.03.18.docx'),
    'chapter4': docx2txt.process(f'{report2_dir}/Chapter4 - Reducing dependencies and increase resilience.docx'),
    'chapter5': docx2txt.process(f'{report2_dir}/Chapter5 - Closing Innovation Gaps.docx'),
    'chapter6': docx2txt.process(f'{report2_dir}/Chapter6 - Financing the green transition.docx'),
    'chapter7': docx2txt.process(f'{report2_dir}/Chapter7 - Fair and just transition.docx'),
    'chapter8': docx2txt.process(f'{report2_dir}/Chapter8 - Horizontal enablers.docx'),
    'chapter1_short': docx2txt.process(f'{report2_dir}/Chapter1 - Setting the scene - SHORT.docx'),
    'chapter2_short': docx2txt.process(f'{report2_dir}/Chapter2 - Challenges and enablers - SHORT.docx'),
    'chapter3_short': docx2txt.process(f'{report2_dir}/Chapter3 - Environmental impacts of future scenarios - RP 2025.03.18 - SHORT.docx'),
    'chapter4_short': docx2txt.process(f'{report2_dir}/Chapter4 - Reducing dependencies and increase resilience - SHORT.docx'),
    'chapter5_short': docx2txt.process(f'{report2_dir}/Chapter5 - Closing Innovation Gaps - SHORT.docx'),
    'chapter7_short': docx2txt.process(f'{report2_dir}/Chapter7 - Fair and just transition - SHORT.docx')
}

# Clean unwanted characters and format
from Code.tools import clean_report
report2 = clean_report(report2) 


# Import a detailed description with examples of what the network weights mean, as in Nilssen et al (2016)
from Code.tools import impact_weight_meanings


# Load API functions
from Code.API import get_chat_response, num_tokens_from_string

# Load the function to generate pairs of thematic areas (used in the loop)
from Code.tools import generate_pairs
ta_pairs = generate_pairs(list=['TA1', 'TA2', 'TA3', 'TA4', 'TA5', 'TA6', 'TA7'], # list of thematic areas
                          count_duplicates=True)

# Load function to generate target data dictionaries based for each type of network (target_to_target, subtheme_to_subtheme, policydoc_to_policydoc)
from Code.tools import generate_dict

# Load tool to aggregate result csv files at the end
from Code.tools import aggregate_csv



### Define LLM parameters


sort_by = 'subthemes', 'policydoc', or 'thematic_area' <br/>

* 'subthemes' -> to generate subtheme-to-subtheme network <br/>
* 'policydoc' -> to generate subtheme-to-subtheme network <br/>
* 'thematic_area' -> to generate target-to-target network. <br/>


Variables to update:
* <font color="red">*date*</font>
* <font color="red">*output_dir*</font>

Parameters you can update:
* <font color="orange">*data*</font>
* <font color="orange">*sort_by*</font>
* <font color="orange">*seed*</font>
* <font color="orange">*temperature*</font>
* <font color="orange">*model*</font>



In [6]:
# LLM parameters

data = target_data_150
sort_by= 'subthemes'  # either'subthemes', 'policydoc', or 'thematic_area', depending on whether you want to generate a subtheme-to-subtheme, policydoc-to-policydoc, or target-to-target network.
seed = None 
temperature = 0.1
model = "llama-3.3-70b-instruct"

date = '0420' 
output_dir = f'/Users/giorgiobolchi2/Documents/GitHub/jrc-egd/LLM/Data/Outputs/{date}/'

prompt = f'''
                Data input & Context:
                - List A: first list of European Green Deal (EGD) targets grouped by sub-themes:{sub1}.
                - List B: second list of EGD targets grouped by sub-themes: {sub2}.
                - Report n°2: {report2['chapter1_short']} + {report2['chapter2_short']} + {report2['chapter3_short']} + {report2['chapter4_short']} + {report2['chapter5_short']} ] + {report2['chapter7_short']} ].

                Task: 
                - Get acquainted with the context of Report n°2 as well as the information available about the sub-themes and their respective targets and data in both lists.
                - Assess the potential impacts of implementing the targets grouped within sub-themes from List A on the implementation of targets in sub-themes from List B, considering how the implementation of one sub-theme's targets may facilitate (positively impact) or hinder (negatively impact) the implementation of another sub-theme's targets.


                Answer format: provide your answer as a table in csv format please (separator: ";"), with the following columns:
                - source_subtheme (e.g., GHG Reduction).
                - impact_subtheme (the name of the subtheme that is likely to be positively or negatively affected by the implementation and requirements of the sub-theme in the 'source_subtheme' column).
                - impact_type (positive '+' or negative '-').
                - impact_weight (-3,-2,-1,1,2,3).
                - justification.

                Specifications:
                - The impacts can have different weights, which have the following meanings: {impact_weight_meanings}
                - Only the following sub-themes can be added to the table: {subtheme_perTA_list[ta_pairs[x][0]]} and {subtheme_perTA_list[ta_pairs[x][1]]}.
                - This is crucial: do not invent new sub-themes.
                - Connections can only be made from sub-themes in List A to sub-themes in List B, not the contrary.
                - If some sub-themes do not have any connections at all (i.e., are isolated), do not add any row.
                - One row per connection, if you deem that one sub-theme has an impact on multiple other sub-themes, add as many rows for a same sub-theme as necessary.
                - It is critical that your analysis is based on the context of the report and not just on the semantics of the target contents.
                - This is mandatory: for each sub-theme connection, write 1-2 concise sentences justifying your choice. 
                - Output only the CSV table. Do not include additional commentary.
            '''

### Generate answers

In [ ]:
    # Inputs:
    #     model, seed, temperature: LLM parameters
    #     ta_pairs: a list of pairs of thematic area codes.
    #     target_data_dict: a dictionary containing target data for each thematic area code
    #     report2: a dictionary containing chapters of report 2
    #     subthemes_list: a list of sub-themes for each thematic area code
    #     impact_weight_meanings: a dictionary containing meanings for impact weights

    # Outputs:
    #     A set of 42 CSV files containing generated answers for each pair of thematic area codes, saved in the output_directory
    #     A CSV file containing metadata for the generated answers, saved in the output_directory as {date}_network_metadata.csv



## (Loop tools and formatting)

# Create the 'date' folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True) 

# create empty panda dataframe with the following columns so to gather a bit more data on the responses and ultimately try to assess consistency
answers_metadata = pd.DataFrame(columns=["ta_pairs_nbr",
                                         "ta_pairs_pairs",        
                                         "model",
                                         "seed",
                                         "temperature",
                                         "system_fingerprint", 
                                         "prompt_tokens", 
                                         "completion_tokens"])  

# create a dictionary to access the exact subtheme names (subtheme_perTA_list) for each thematic_area_code
subtheme_perTA_list = data.groupby('thematic_area_code')['sub_theme'].apply(list).to_dict() 

# Generate the target data dictionary sorted by either subthemes, policydoc, or thematic_area depending on what network you want.
target_data_dict = generate_dict(data=data,
                                 sort_by=sort_by)

## Loop

for x in range(len(ta_pairs)):
#for x in range(8, 42):     # for testing purposes
#for x in list([28,34]): # for testing purposes

    success = False  # Initialize a flag to track whether the operation was successful
    retry_count = 0  # Initialize a counter to track the number of retries
    max_retries = 5  # adjust this value to set the desired number of retries

    while not success and retry_count < max_retries:
        try:
            # Subset data to avoid overloading the model
            sub1 = [f"{target_data_dict[ta_pairs[x][0]]}"] # this will access the data stored in target_data_dict for the thematic_area_code stored in ta_pairs[x][0] (e.g., target_data_dict[ta_pairs[0][0]] <=> target_data_dict['TA1])
            sub2 = [f"{target_data_dict[ta_pairs[x][1]]}"] # same thing here, but for the second element of the pair.


            # Define prompt
            prompt = f'''
                Data input & Context:
                - List A: first list of European Green Deal (EGD) targets grouped by sub-themes:{sub1}.
                - List B: second list of EGD targets grouped by sub-themes: {sub2}.
                - Report n°2: {report2['chapter1_short']} + {report2['chapter2_short']} + {report2['chapter3_short']} + {report2['chapter4_short']} + {report2['chapter5_short']} ] + {report2['chapter7_short']} ].

                Task: 
                - Get acquainted with the context of Report n°2 as well as the information available about the sub-themes and their respective targets and data in both lists.
                - Assess the potential impacts of implementing the targets grouped within sub-themes from List A on the implementation of targets in sub-themes from List B, considering how the implementation of one sub-theme's targets may facilitate (positively impact) or hinder (negatively impact) the implementation of another sub-theme's targets.


                Answer format: provide your answer as a table in csv format please (separator: ";"), with the following columns:
                - source_subtheme (e.g., GHG Reduction).
                - impact_subtheme (the name of the subtheme that is likely to be positively or negatively affected by the implementation and requirements of the sub-theme in the 'source_subtheme' column).
                - impact_type (positive '+' or negative '-').
                - impact_weight (-3,-2,-1,1,2,3).
                - justification.

                Specifications:
                - The impacts can have different weights, which have the following meanings: {impact_weight_meanings}
                - Only the following sub-themes can be added to the table: {subtheme_perTA_list[ta_pairs[x][0]]} and {subtheme_perTA_list[ta_pairs[x][1]]}.
                - This is crucial: do not invent new sub-themes.
                - Connections can only be made from sub-themes in List A to sub-themes in List B, not the contrary.
                - If some sub-themes do not have any connections at all (i.e., are isolated), do not add any row.
                - One row per connection, if you deem that one sub-theme has an impact on multiple other sub-themes, add as many rows for a same sub-theme as necessary.
                - It is critical that your analysis is based on the context of the report and not just on the semantics of the target contents.
                - This is mandatory: for each sub-theme connection, write 1-2 concise sentences justifying your choice. 
                - Output only the CSV table. Do not include additional commentary.
            '''

            # Print pre-generation metadata (to double check amount of tokens in prompt, JRC llama3.3 should have a max of 120k)
            prompt_metadata = f'''TA_pair: {x} - {ta_pairs[x]} \nPrompt length: {len(prompt)} \nPrompt tokens (o200k_base encoding): {num_tokens_from_string(prompt, "o200k_base")} \nPrompt tokens (cl100k_base encoding): {num_tokens_from_string(prompt, "cl100k_base")} \n'''
            print(prompt_metadata)

            # Generate answer
            answer = get_chat_response(prompt=prompt,
                                      seed=seed,
                                      model=model,
                                      temperature=temperature)

            # Print post-generation metadata 
            print(f'Prompt tokens: {answer["prompt_tokens"]} \nCompletion tokens: {answer["completion_tokens"]}')


            # Add the metadata of the generated answer to the previously created dataframe
            answers_metadata.loc[x] = (x,
                                       ta_pairs[x],
                                       model,
                                       seed,
                                       temperature,
                                       answer["system_fingerprint"],
                                       answer["prompt_tokens"],
                                       answer["completion_tokens"])

            # Save the generated answer as a CSV file
            output_name = f'{date}_network_pair{x}.csv'

            with open((os.path.join(output_dir, output_name)), 'w') as f:
                f.write(answer["response_content"])

            # If success, set the success flag to True
            success = True

            # If success, add a 1-minute pause between answer requests to avoid RateLimitErrors
            print(f"-- 1 min pause \n")
            time.sleep(60)

        # Manage errors
        except Exception as e: 
            retry_count += 1  # Increment the retry counter if an error occurs
            error_type = type(e).__name__  # Get the type of error that occurred
            error_message = str(e) # Get the error message
            print(f"An error occurred ({error_type}): {error_message}. Retrying ({retry_count}/{max_retries})")  # Print an error message with the type and message

    # Print a message if the operation failed after the maximum number of retries
    if not success:
        print(f"Failed to generate answer for pair{x} ({ta_pairs[x]}) after {max_retries} retries.") 

# Save the metadata dataframe as a CSV file
answers_metadata.to_csv(path_or_buf=os.path.join(output_dir, f'{date}_network_metadata.csv'), 
                         sep=';',
                         index=False)


In [12]:

# Aggregate all CSV files in the output directory

aggregate_csv(date= date,
              output_dir= output_dir,
              sep=';',
              file_pattern= f'{date}_network_pair*.csv')

